# Week 5 · Day 2 — LangChain: Tools, Chains, Memory & Your First Framework Agent
### Groq API edition

**Note on API provider:** The brief references `langchain-anthropic`. This notebook uses
**`langchain-groq`** instead, so the agent runs on the Groq API. LangChain's abstractions
(`Tool`, `AgentExecutor`, Memory, LCEL) are provider-agnostic — swapping the chat model
class is the only real change.

**Setup**

```bash
pip install langchain langchain-groq langchain-core langchain-classic python-dotenv pydantic
```

> **Version note:** as of LangChain 1.0 (late 2025), `create_tool_calling_agent` and
> `AgentExecutor` were moved out of the main `langchain` package into a companion
> package, **`langchain-classic`** (LangChain now recommends the newer LangGraph-based
> `create_agent` for new projects, but this notebook uses the classic constructors named
> in the brief). If `pip install langchain` gave you 1.x, `from langchain.agents import
> create_tool_calling_agent` will raise `ImportError` — install `langchain-classic` and
> import from there instead. The cell below handles both cases automatically.

Create a `.env` file in the same folder as this notebook:
```
GROQ_API_KEY=gsk_your_key_here
```


In [1]:
import os
import json
import random
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env into the environment

from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"  # same model family as Day 1, swap if deprecated

llm = ChatGroq(
    model=MODEL,
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
)


---
## Task 1 — LangChain Setup & Core Concepts

### Mapping Day 1 (raw Python) → LangChain

| Day 1 raw-Python concept | LangChain equivalent |
|---|---|
| `client.chat.completions.create(...)` call | **LLM wrapper** — `ChatGroq(...)`, a `Runnable` object with a standard `.invoke()` interface |
| Hand-written `TOOLS` list of JSON schemas + `TOOL_FUNCTIONS` dispatch dict | **`Tool`** objects — created via the `@tool` decorator, which derives the schema from the function signature + docstring instead of a hand-written dict |
| The `for` loop in `run_agent()` (send → check tool_calls → execute → append → repeat) | **`AgentExecutor`** — runs the same loop internally, calling the agent, executing chosen tools, and feeding results back, until a final answer or `max_iterations` |
| The growing `messages` list passed on every call | **Memory** — `ConversationBufferMemory` / `RunnableWithMessageHistory`, which manages that list for you and can persist it across separate `.invoke()` calls |

### A basic LCEL pipeline

LCEL (LangChain Expression Language) lets you compose a prompt template, a model, and an
output parser into a single pipeline using the `|` operator.


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in exactly two sentences for a beginner data science student."
)
output_parser = StrOutputParser()

# LCEL: prompt -> llm -> parser, wired with the pipe operator
chain = prompt | llm | output_parser

result = chain.invoke({"topic": "the ReAct agent pattern"})
print(result)


The ReAct agent pattern is a design approach that allows you to create autonomous agents that can react to changing conditions in a system, making decisions based on their current state and the state of their environment. In the context of data science, ReAct can be used to build adaptive systems that learn from data and make decisions in real-time, such as recommender systems, chatbots, or autonomous vehicles, by breaking down complex problems into smaller, manageable components that interact with each other.


**What `|` is doing under the hood:** every LangChain component (`ChatPromptTemplate`,
`ChatGroq`, `StrOutputParser`) implements the same `Runnable` interface (`.invoke()`,
`.batch()`, `.stream()`). The `|` operator is Python's overloadable `__or__` method,
overridden here to wrap two runnables into a `RunnableSequence` — calling `.invoke(x)` on
the sequence just calls the first runnable's `.invoke(x)`, passes its output as the input
to the next runnable's `.invoke()`, and so on down the chain. It's function composition
with a readable syntax, not a new execution model.


---
## Task 2 — Define & Register Tools

Three tools below, built with the `@tool` decorator: two carried over from Day 1
(`calculator`, `get_weather`) and one new tool, `lookup_price`, that reads from a real
local JSON "database" file on disk (standing in for a product-pricing API).


In [3]:
# --- Build a tiny local JSON "database" of product prices (our real data source) ---
PRODUCTS_PATH = "products.json"

products_db = {
    "laptop a": {"price_usd": 799, "category": "laptop", "specs": "8GB RAM, 256GB SSD"},
    "laptop b": {"price_usd": 1199, "category": "laptop", "specs": "16GB RAM, 512GB SSD"},
    "laptop c": {"price_usd": 549, "category": "laptop", "specs": "8GB RAM, 128GB SSD"},
    "phone x":  {"price_usd": 699, "category": "phone",  "specs": "6.1in, 128GB"},
    "phone y":  {"price_usd": 999, "category": "phone",  "specs": "6.7in, 256GB"},
}

with open(PRODUCTS_PATH, "w") as f:
    json.dump(products_db, f, indent=2)

print(f"Wrote {PRODUCTS_PATH} with {len(products_db)} products.")


Wrote products.json with 5 products.


In [4]:
from langchain_core.tools import tool

# --- Tool 1: calculator (carried over from Day 1) --------------------------
@tool
def calculator(expression: str) -> str:
    """Evaluates a single arithmetic expression and returns the numeric result.

    Supports +, -, *, /, **, and parentheses. Use this whenever a request needs a
    precise numeric computation instead of an estimate. Pass numbers and operators
    only -- e.g. '(799 + 1199) / 2' -- never words or variable names.
    """
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: could not evaluate expression -- {e}"


# --- Tool 2: weather lookup stub (carried over from Day 1) -----------------
_FAKE_WEATHER_DB = {
    "lahore": {"condition": "Sunny", "temp_c": 39},
    "islamabad": {"condition": "Partly cloudy", "temp_c": 33},
    "london": {"condition": "Rainy", "temp_c": 18},
    "new york": {"condition": "Clear", "temp_c": 27},
}

@tool
def get_weather(city: str) -> str:
    """Looks up the current weather condition and temperature (Celsius) for a named city.

    Only works for cities in the demo database (Lahore, Islamabad, London, New York).
    Use this whenever a request asks about current weather or temperature, or wants a
    comparison between cities. Returns an error message for unknown cities.
    """
    key = city.strip().lower()
    if key not in _FAKE_WEATHER_DB:
        return f"Error: no weather data for '{city}'."
    data = _FAKE_WEATHER_DB[key]
    return f"{city}: {data['condition']}, {data['temp_c']}C"


# --- Tool 3: NEW -- reads from a real local JSON data source ---------------
@tool
def lookup_price(product_name: str) -> str:
    """Looks up the price (USD) and specs of a product from the product catalog.

    The catalog is read fresh from products.json on every call. Use this whenever a
    request asks for the price of a named product, or wants to compare prices between
    two or more named products. product_name should match a catalog entry such as
    'Laptop A', 'Laptop B', 'Laptop C', 'Phone X', or 'Phone Y' (case-insensitive).
    Returns an error message if the product isn't found in the catalog.
    """
    with open(PRODUCTS_PATH) as f:
        db = json.load(f)
    key = product_name.strip().lower()
    if key not in db:
        return f"Error: '{product_name}' not found in catalog. Known products: {list(db.keys())}"
    entry = db[key]
    return f"{product_name}: ${entry['price_usd']} ({entry['specs']})"


TOOLS = [calculator, get_weather, lookup_price]
for t in TOOLS:
    print(t.name, "->", t.description[:70].replace("\n", " "), "...")


calculator -> Evaluates a single arithmetic expression and returns the numeric resul ...
get_weather -> Looks up the current weather condition and temperature (Celsius) for a ...
lookup_price -> Looks up the price (USD) and specs of a product from the product catal ...


**Why the docstring matters:** with the `@tool` decorator, LangChain builds the tool's
JSON schema automatically from the function signature (for argument names/types) and its
**docstring** (for the `description` field sent to the model). The docstring isn't just
documentation for a human reader anymore — it becomes part of the prompt the LLM sees when
deciding whether and how to call the tool. This is the same principle as Day 1's hand-written
`description` fields, just derived automatically instead of typed into a dict by hand: a
vague docstring ("looks up a product") produces exactly the same mis-firing/mis-argument
problems a vague JSON schema description would.


---
## Task 3 — Build an Agent with `create_tool_calling_agent` / `AgentExecutor`


In [5]:
# LangChain >=1.0 moved these constructors into the langchain-classic package.
# This works whether you're on legacy (<1.0, symbols live in langchain.agents)
# or current (>=1.0, symbols live in langchain_classic.agents) LangChain.
try:
    from langchain.agents import create_tool_calling_agent, AgentExecutor
except ImportError:
    from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

from langchain_core.prompts import MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. Use them whenever "
               "they would give you more accurate information than guessing."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, TOOLS, agent_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,       # prints the full Thought/Action/Observation trace
    max_iterations=6,   # same safeguard as Day 1's max_iterations
    handle_parsing_errors=True,  # gracefully recover from malformed tool-call output
)


In [6]:
# Multi-step run: requires 2+ tool calls (price lookup x2) plus reasoning over the result.
response = agent_executor.invoke({
    "input": "Find the price of Laptop A and Laptop B, then tell me which one is "
             "the better deal per dollar of RAM."
})
print("\n=== FINAL ANSWER ===")
print(response["output"])




> Entering new AgentExecutor chain...

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`


Laptop A: $799 (8GB RAM, 256GB SSD)
Invoking: `lookup_price` with `{'product_name': 'Laptop B'}`


Laptop B: $1199 (16GB RAM, 512GB SSD)
Invoking: `calculator` with `{'expression': '799 / 8'}`


99.875
Invoking: `calculator` with `{'expression': '1199 / 16'}`


74.9375Laptop A costs $99.875 per GB of RAM and Laptop B costs $74.9375 per GB of RAM. Therefore, Laptop B is the better deal per dollar of RAM.

> Finished chain.

=== FINAL ANSWER ===
Laptop A costs $99.875 per GB of RAM and Laptop B costs $74.9375 per GB of RAM. Therefore, Laptop B is the better deal per dollar of RAM.


### Annotated reasoning trace

With `verbose=True`, `AgentExecutor` prints a trace to stdout that looks like this
(annotated against the ReAct phases from Day 1):

```
> Entering new AgentExecutor chain...              <- loop starts

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`   <- REASON + ACT
Laptop A: $799 (8GB RAM, 256GB SSD)                              <- OBSERVE

Invoking: `lookup_price` with `{'product_name': 'Laptop B'}`   <- REASON + ACT
Laptop B: $1199 (16GB RAM, 512GB SSD)                             <- OBSERVE

Laptop A costs $799 for 8GB RAM (~$99.9/GB); Laptop B costs         <- REASON
$1199 for 16GB RAM (~$74.9/GB). Laptop B is the better deal
per dollar of RAM.
> Finished chain.                                    <- loop ends, no more tool calls
```

### Day 1 raw-Python log vs. this trace — what's similar, what's hidden

**Similar:** the underlying shape is identical — reason, pick a tool, call it, observe the
result, reason again, eventually stop. It's the exact same ReAct loop from Day 1, just
executed by `AgentExecutor` instead of our own `for` loop.

**Hidden now:**
- The raw `messages` list (system/human/assistant/tool roles) is no longer visible by
  default — `AgentExecutor` builds and mutates it internally via the `agent_scratchpad`.
- The exact moment arguments are JSON-parsed, and what happens if parsing fails, is
  handled inside LangChain's agent runtime — we only see the outcome via
  `handle_parsing_errors=True`, not the parsing code itself.
- The stop condition ("no more tool_calls in the response") is the same rule we wrote
  by hand on Day 1, but it now lives inside `create_tool_calling_agent`'s output parser,
  not in code we can step through directly.


---
## Task 4 — Add Memory

Wiring `agent_executor` with `RunnableWithMessageHistory` so it can handle a 3-turn
conversation where later turns depend on earlier context (e.g. "compare it to Y" needs to
remember what "it" was).


In [7]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# One in-memory history store per session_id -- swap for a persistent store
# (Redis, a DB table, etc.) in a real deployment.
_session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


C:\Users\Haroon\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
config = {"configurable": {"session_id": "budget-client-demo"}}

turn1 = agent_with_memory.invoke(
    {"input": "Find the price of Laptop A."}, config=config
)
print("Turn 1:", turn1["output"])

turn2 = agent_with_memory.invoke(
    {"input": "Now compare it to Laptop C."}, config=config
)
print("\nTurn 2:", turn2["output"])

turn3 = agent_with_memory.invoke(
    {"input": "Which one should I recommend to a budget-conscious client?"}, config=config
)
print("\nTurn 3:", turn3["output"])




> Entering new AgentExecutor chain...

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`


Laptop A: $799 (8GB RAM, 256GB SSD)The price of Laptop A is $799. It has 8GB RAM and a 256GB SSD.

> Finished chain.
Turn 1: The price of Laptop A is $799. It has 8GB RAM and a 256GB SSD.


> Entering new AgentExecutor chain...

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`


Laptop A: $799 (8GB RAM, 256GB SSD)
Invoking: `lookup_price` with `{'product_name': 'Laptop C'}`


Laptop C: $549 (8GB RAM, 128GB SSD)Laptop A costs $250 more than Laptop C and has twice the storage.

> Finished chain.

Turn 2: Laptop A costs $250 more than Laptop C and has twice the storage.


> Entering new AgentExecutor chain...

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`


Laptop A: $799 (8GB RAM, 256GB SSD)
Invoking: `lookup_price` with `{'product_name': 'Laptop C'}`


Laptop C: $549 (8GB RAM, 128GB SSD)Based on the prices, I would recommend Laptop C to a budget-conscious cl

Turn 2 ("compare **it** to Laptop C") and turn 3 ("**which one**...") only work because
`chat_history` carries turn 1's result forward automatically — the agent never has to be
re-told what "it" refers to. This is the direct framework equivalent of manually appending
every message to Day 1's `messages` list; here it's handled by `RunnableWithMessageHistory`
keyed off `session_id`.


---
## Task 5 — Structured Output & Error Handling

### Structured final answer via a Pydantic model


In [9]:
from pydantic import BaseModel, Field
from typing import Optional

class RecommendationOutput(BaseModel):
    recommended_product: str = Field(description="Name of the recommended product")
    price_usd: float = Field(description="Price of the recommended product in USD")
    alternative_product: Optional[str] = Field(
        default=None, description="Name of the product it was compared against"
    )
    reasoning: str = Field(description="One or two sentence justification for the pick")


structured_llm = llm.with_structured_output(RecommendationOutput)

# Feed the structured model the agent's already-gathered findings (its own final
# answer text) and ask it to package that into the schema -- a common pattern:
# let the tool-calling agent do the research, then a second structured-output
# call turns the free-text result into a validated object.
structured_prompt = ChatPromptTemplate.from_template(
    "Based on this analysis, extract a structured recommendation:\n\n{analysis}"
)
structured_chain = structured_prompt | structured_llm

structured_result = structured_chain.invoke({"analysis": turn3["output"]})
print(structured_result)
print(type(structured_result))


recommended_product='Laptop C' price_usd=250.0 alternative_product='Laptop A' reasoning='It has similar specs to Laptop A but is $250 cheaper, with the only difference being half the storage.'
<class '__main__.RecommendationOutput'>


### Error handling: a tool that sometimes fails


In [10]:
@tool
def check_stock(product_name: str) -> str:
    """Checks whether a product is currently in stock.

    Use this after finding a product's price, if the user asks about availability.
    This demo tool randomly simulates a backend outage about 1 in 3 calls -- if it
    fails, report that stock status is temporarily unavailable rather than guessing.
    """
    if random.random() < 0.33:
        raise RuntimeError("Inventory service timed out (simulated failure).")
    return f"{product_name} is in stock."


tools_with_flaky = TOOLS + [check_stock]

flaky_agent = create_tool_calling_agent(llm, tools_with_flaky, agent_prompt)
flaky_executor = AgentExecutor(
    agent=flaky_agent,
    tools=tools_with_flaky,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True,
    # This is the key setting: without it, a raised exception inside a tool
    # propagates up and kills the whole AgentExecutor run. With it, LangChain
    # catches the exception, turns it into an Observation string, and lets the
    # agent reason about the failure and recover (e.g. retry, or tell the user).
    handle_tool_error=True,
)

result = flaky_executor.invoke({
    "input": "Is Laptop A in stock, and if so what's the price?"
})
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `lookup_price` with `{'product_name': 'Laptop A'}`


Laptop A: $799 (8GB RAM, 256GB SSD)
Invoking: `check_stock` with `{'product_name': 'Laptop A'}`


Laptop A is in stock.The price of Laptop A is $799, and it is in stock.

> Finished chain.
The price of Laptop A is $799, and it is in stock.


**How the agent recovers:** with `handle_tool_error=True` on the `AgentExecutor` (or
alternatively `handle_tool_error=` set per-tool), a raised exception inside `check_stock`
is caught and converted into a plain-text Observation (something like `"Error: Inventory
service timed out (simulated failure)."`) that gets fed back into the loop just like a
normal tool result. The agent then reasons over that observation on its next step — it
might retry the call, fall back to reporting the price without stock info, or apologize
and say availability couldn't be confirmed. Without `handle_tool_error=True`, the raised
`RuntimeError` propagates straight up and crashes the entire `AgentExecutor.invoke()` call,
the same way an unhandled exception would crash Day 1's raw loop if `TOOL_FUNCTIONS`
weren't wrapped in `try/except`.

### What LangChain made easier vs. Day 1, and where the "magic" shows

LangChain removed most of the boilerplate: tool schemas are inferred from function
signatures/docstrings instead of hand-typed JSON, the reasoning loop is `AgentExecutor`
instead of a hand-rolled `while`, multi-turn memory is a few lines of
`RunnableWithMessageHistory` instead of manually managing a list, and structured output is
one `.with_structured_output()` call instead of hand-parsing JSON. The leakiness shows up
exactly where the raw version made everything explicit: the `agent_scratchpad` message
list is built and mutated internally, so debugging a bad tool call means reading a
`verbose=True` trace rather than stepping through code; `handle_parsing_errors` and
`handle_tool_error` are flags whose *exact* recovery behavior (what text gets shown to the
model, how many retries) isn't visible unless you go read LangChain's source — on Day 1 we
wrote and could see that exact string ourselves.
